In [1]:
!pip install torch -q

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [3]:
pairs = [
    ("hello", "hi how are you"),
    ("how are you", "i am fine"),
    ("what is your name", "i am a chatbot"),
    ("bye", "goodbye"),
    ("thanks", "you are welcome")
]

In [4]:
word2idx = {"<pad>":0, "<sos>":1, "<eos>":2}
idx2word = {0:"<pad>", 1:"<sos>", 2:"<eos>"}

def build_vocab(pairs):
    idx = 3
    for q, a in pairs:
        for word in (q + " " + a).split():
            if word not in word2idx:
                word2idx[word] = idx
                idx2word[idx] = word
                idx += 1

build_vocab(pairs)
vocab_size = len(word2idx)

In [5]:
def encode(sentence):
    return [word2idx[w] for w in sentence.split()] + [word2idx["<eos>"]]

In [10]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size*2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[0]
        hidden = hidden.repeat(seq_len, 1, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=0)

In [11]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_size=64, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.encoder = nn.LSTM(emb_size, hidden_size)
        self.decoder = nn.LSTM(emb_size, hidden_size)
        self.attention = Attention(hidden_size)
        self.fc = nn.Linear(hidden_size*2, vocab_size)

    def forward(self, src, trg):
        embedded = self.embedding(src)
        encoder_outputs, (hidden, cell) = self.encoder(embedded)

        outputs = []
        input = trg[0]

        for t in range(1, len(trg)):
            embedded = self.embedding(input).unsqueeze(0)
            output, (hidden, cell) = self.decoder(embedded, (hidden, cell))

            attn_weights = self.attention(hidden, encoder_outputs)
            context = torch.sum(attn_weights.unsqueeze(2) * encoder_outputs, dim=0)

            output = torch.cat((output.squeeze(0), context), dim=1)
            prediction = self.fc(output)

            outputs.append(prediction)
            input = trg[t]

        # ✅ FIX: avoid empty stack error
        if len(outputs) == 0:
            return torch.zeros(1, 1, vocab_size)

        return torch.stack(outputs)

In [12]:
model = Seq2Seq(vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(200):
    total_loss = 0

    for q, a in pairs:
        src = torch.tensor(encode(q)).unsqueeze(1)
        trg = torch.tensor([word2idx["<sos>"]] + encode(a)).unsqueeze(1)

        optimizer.zero_grad()
        output = model(src, trg)

        output = output.view(-1, vocab_size)
        trg = trg[1:].view(-1)

        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 15.2205
Epoch 20, Loss: 0.4526
Epoch 40, Loss: 0.0988
Epoch 60, Loss: 0.0468
Epoch 80, Loss: 0.0283
Epoch 100, Loss: 0.0192
Epoch 120, Loss: 0.0138
Epoch 140, Loss: 0.0103
Epoch 160, Loss: 0.0082
Epoch 180, Loss: 0.0066


In [13]:
def generate_reply(sentence):
    model.eval()

    src = torch.tensor(encode(sentence)).unsqueeze(1)
    trg = torch.tensor([word2idx["<sos>"]]).unsqueeze(1)

    words = []

    for _ in range(10):
        output = model(src, trg)

        # ✅ FIX: handle empty output
        if output.shape[0] == 0:
            break

        pred = output[-1].argmax().item()

        if pred == word2idx["<eos>"]:
            break

        words.append(idx2word[pred])

        trg = torch.cat([trg, torch.tensor([[pred]])], dim=0)

    return " ".join(words)